# Résultats des évaluations - "manuelles" & ragas

### Librairies nécessaires

In [1]:
import pandas as pd

## Evaluation des questions/réponses de manière "manuelle"
L'objectif ici a été de générer un score de pertinence afin de déterminer le bon seuil entre nos données et nous permettre d'évaluer de manière "manuelle" si les réponses sont pertinentes ou non.

### On charge le csv de notre évaluation

In [17]:
df_ragas_ma = pd.read_csv("../data/resultat_evaluation.csv", sep=";")

#### Regardons les résultats de notre évaluation humaine au global

In [42]:
df_ragas_ma['Eval Humaine'].value_counts(normalize=True)

Eval Humaine
Correcte                  0.63
Incorrecte                0.19
Partiellement correcte    0.18
Name: proportion, dtype: float64

- On voit une majorité de réponse correcte, ce qui est un bon début

#### Regardons les résultats de notre évaluation humaine sur les questions factuelles (réponses exactes se trouvent dans le jeu de données)

In [43]:
num_questions_five = ['Question 1 :','Question 2 :','Question 3 :','Question 4 :','Question 5 :']
df_ragas_ma_five = df_ragas_ma[df_ragas_ma['Numéro question'].isin(num_questions_five)]
df_ragas_ma_five['Eval Humaine'].value_counts(normalize=True)

Eval Humaine
Correcte      0.88
Incorrecte    0.12
Name: proportion, dtype: float64

- On voit que notre évaluation remonte drastiquement en prenant uniquement des questions à laquelle le chatbot doit avoir juste. Les 12% d'erreur sont liés aux différents tests de paramétrages effectués (nous avons testé volontairement des paramètres agressifs)

## Evaluation avec Ragas - Analyse des résultats
Ragas est un framework qui permet d'évaluer les performances d'un pipeline RAG à travers plusieurs métriques.

### On charge le csv de notre évaluation ragas

In [9]:
df_ragas = pd.read_csv("../data/evaluation_ragas.csv", sep=";")

### Petit check des colonnes pour les choisir correctement

In [45]:
df_ragas.columns

Index(['index', 'Numéro question', 'Question', 'Réponse Attendue',
       'Réponse LLM', 'Score Qualité (LLM)',
       'Documents & Scores Distance (Pour le Seuil)',
       'Context tiré des métadonnées', 'Seuil utilisé min',
       'Nombre de documents utilisés max', 'Eval Humaine', 'contexts',
       'faithfulness', 'answer_relevancy', 'context_precision',
       'context_recall'],
      dtype='object')

## Les résultats de l'évaluation
- Prendre les résultats globaux du fichier n'ont pas trop de sens car nous analysons plusieurs seuils et un nombre de documents différents.
- Nous allons analyser les résultats moyens des métriques utilisées par type de paramétrage testé.
- **Les métriques utilisées** :
    - **faithfulness**, est-ce que la génération est fidèle au contexte ?
    - **answer_relevancy**, est-ce que la génération de la réponse est pertinente à la question ?
    - **context_precision**, est-ce que la récupération du contexte est précise (peu de bruit) ?
    - **context_recall**, est-ce que la récupération des infos clés sont correctements récupérées ?


### On filtre le dataframe généré en fonction des paramètres utilisés et du type d'index (long ou court)
- Fonction pour générer les résultats.
- **Sur les 20 questions, il faut tenir comptes des questions volontairement ambigües qui baisse le résultat de certaines métriques.**
- Meilleurs paramétrages ici : k = 3 et seuil à 0.50 et dans une moindre mesure k = 3 et seuil à 0.65
- On ajoute le % des résultats de l'évaluation huamine pour pouvoir comparer les 2
- On voit bien une logique entre les résultats de ragas et l'évaluation humaine (hausse de % correcte quand on analyse les questions factuelles par exemple)
##
- Sur les 20 questions, les paramètres k=3 et seuil=0.50 paraît être meilleur
- Cependant sur les questions factuelles(5 premières questions), **les résultats sont meilleurs sur les paramètres k=3 et seuil à 0.65**
- On peut partir sur les paramètres k=3 et seuil à 0.65 afin d'avoir des documents plus "similaires" les uns des autres.
- **Et nous prenons la base de FAISS version courte car cela va nous permettre de générer moins de token dans un premier temps et les réponses sont pour le moment en adéquation avec ce que l'on recherche pour ce POC.**

In [34]:
# Fonction pour générer les résultats
def resultats_rag(seuil, doc, index):
    filtered = df_ragas[
        (df_ragas["Seuil utilisé min"] == seuil) &
        (df_ragas["Nombre de documents utilisés max"] == doc) &
        (df_ragas["index"] == index)]
    print(f"\n===== Résultats de l'évaluation ragas avec un seuil à {seuil}, k = {doc} et la version {index} =====")
    print(f"Résultat du faithfullness : {round(filtered['faithfulness'].mean(),4)}")
    print(f"Résultat du answer_relevancy : {round(filtered['answer_relevancy'].mean(),4)}")
    print(f"Résultat du context_precision : {round(filtered['context_precision'].mean(),4)}")
    print(f"Résultat du context_recall : {round(filtered['context_recall'].mean(),4)}")
    print(f"\n% de réponses correctes : {round(filtered['Eval Humaine'].value_counts(normalize=True),4)}")

    return

In [35]:
# On applique la fonction en fonction des différents paramètres
resultats_rag(0.50, 3, "faiss_index_short")
resultats_rag(0.80, 3, "faiss_index_short")
resultats_rag(0.65, 3, "faiss_index_short")
resultats_rag(0.65, 3, "faiss_index_long")
resultats_rag(0.70, 5, "faiss_index_short")



===== Résultats de l'évaluation ragas avec un seuil à 0.5, k = 3 et la version faiss_index_short =====
Résultat du faithfullness : 0.8794
Résultat du answer_relevancy : 0.7114
Résultat du context_precision : 0.5614
Résultat du context_recall : 0.7167

% de réponses correctes : Eval Humaine
Correcte                  0.75
Partiellement correcte    0.15
Incorrecte                0.10
Name: proportion, dtype: float64

===== Résultats de l'évaluation ragas avec un seuil à 0.8, k = 3 et la version faiss_index_short =====
Résultat du faithfullness : 0.5658
Résultat du answer_relevancy : 0.6176
Résultat du context_precision : 0.2
Résultat du context_recall : 0.5

% de réponses correctes : Eval Humaine
Correcte                  0.5
Incorrecte                0.4
Partiellement correcte    0.1
Name: proportion, dtype: float64

===== Résultats de l'évaluation ragas avec un seuil à 0.65, k = 3 et la version faiss_index_short =====
Résultat du faithfullness : 0.8861
Résultat du answer_relevancy : 0.

### On génère de nouveaux les résultats mais avec uniquement les questions factuelles (5 premières questions)

In [36]:
# Fonction pour générer les résultats
num_questions_five = ['Question 1 :','Question 2 :','Question 3 :','Question 4 :','Question 5 :']
df_questions_five = df_ragas[df_ragas['Numéro question'].isin(num_questions_five)]
def resultats_rag_five(seuil, doc, index):
    filtered_five = df_questions_five[
        (df_questions_five["Seuil utilisé min"] == seuil) &
        (df_questions_five["Nombre de documents utilisés max"] == doc) &
        (df_questions_five["index"] == index)]
    print(f"\n===== Résultats de l'évaluation ragas avec un seuil à {seuil}, k = {doc} et la version {index} =====")
    print(f"Résultat du faithfullness : {round(filtered_five['faithfulness'].mean(),4)}")
    print(f"Résultat du answer_relevancy : {round(filtered_five['answer_relevancy'].mean(),4)}")
    print(f"Résultat du context_precision : {round(filtered_five['context_precision'].mean(),4)}")
    print(f"Résultat du context_recall : {round(filtered_five['context_recall'].mean(),4)}")
    print(f"\n% de réponses correctes : {round(filtered_five['Eval Humaine'].value_counts(normalize=True),4)}")

    return

In [37]:
# On applique la fonction en fonction des différents paramètres
resultats_rag_five(0.50, 3, "faiss_index_short")
resultats_rag_five(0.80, 3, "faiss_index_short")
resultats_rag_five(0.65, 3, "faiss_index_short")
resultats_rag_five(0.65, 3, "faiss_index_long")
resultats_rag_five(0.70, 5, "faiss_index_short")


===== Résultats de l'évaluation ragas avec un seuil à 0.5, k = 3 et la version faiss_index_short =====
Résultat du faithfullness : 1.0
Résultat du answer_relevancy : 0.9068
Résultat du context_precision : 0.9667
Résultat du context_recall : 1.0

% de réponses correctes : Eval Humaine
Correcte    1.0
Name: proportion, dtype: float64

===== Résultats de l'évaluation ragas avec un seuil à 0.8, k = 3 et la version faiss_index_short =====
Résultat du faithfullness : 0.6
Résultat du answer_relevancy : 0.722
Résultat du context_precision : 0.6
Résultat du context_recall : 0.6

% de réponses correctes : Eval Humaine
Correcte      0.6
Incorrecte    0.4
Name: proportion, dtype: float64

===== Résultats de l'évaluation ragas avec un seuil à 0.65, k = 3 et la version faiss_index_short =====
Résultat du faithfullness : 1.0
Résultat du answer_relevancy : 0.9228
Résultat du context_precision : 0.9667
Résultat du context_recall : 1.0

% de réponses correctes : Eval Humaine
Correcte    1.0
Name: propo

### On génère de nouveaux les résultats mais avec uniquement les questions factuelles et des questions avec réponses multiples (10 premières questions)

In [15]:
# Fonction pour générer les résultats
num_questions_ten = ['Question 1 :','Question 2 :','Question 3 :','Question 4 :','Question 5 :', 'Question 6 :','Question 7 :','Question 8 :','Question 9 :','Question 10 :']
df_questions_ten = df_ragas[df_ragas['Numéro question'].isin(num_questions_ten)]
def resultats_rag_ten(seuil, doc, index):
    filtered_ten = df_questions_ten[
        (df_questions_ten["Seuil utilisé min"] == seuil) &
        (df_questions_ten["Nombre de documents utilisés max"] == doc) &
        (df_questions_ten["index"] == index)]
    print(f"\n===== Résultats de l'évaluation ragas avec un seuil à {seuil}, k = {doc} et la version {index} =====")
    print(f"Résultat du faithfullness : {round(filtered_ten['faithfulness'].mean(),4)}")
    print(f"Résultat du answer_relevancy : {round(filtered_ten['answer_relevancy'].mean(),4)}")
    print(f"Résultat du context_precision : {round(filtered_ten['context_precision'].mean(),4)}")
    print(f"Résultat du context_recall : {round(filtered_ten['context_recall'].mean(),4)}")
    print(f"\n% de réponses correctes : {round(filtered_ten['Eval Humaine'].value_counts(normalize=True),4)}")

    return

In [39]:
# On applique la fonction en fonction des différents paramètres
resultats_rag_ten(0.50, 3, "faiss_index_short")
resultats_rag_ten(0.80, 3, "faiss_index_short")
resultats_rag_ten(0.65, 3, "faiss_index_short")
resultats_rag_ten(0.65, 3, "faiss_index_long")
resultats_rag_ten(0.70, 5, "faiss_index_short")


===== Résultats de l'évaluation ragas avec un seuil à 0.5, k = 3 et la version faiss_index_short =====
Résultat du faithfullness : 0.8875
Résultat du answer_relevancy : 0.8854
Résultat du context_precision : 0.6833
Résultat du context_recall : 0.8333

% de réponses correctes : Eval Humaine
Correcte                  0.8
Partiellement correcte    0.1
Incorrecte                0.1
Name: proportion, dtype: float64

===== Résultats de l'évaluation ragas avec un seuil à 0.8, k = 3 et la version faiss_index_short =====
Résultat du faithfullness : 0.525
Résultat du answer_relevancy : 0.6488
Résultat du context_precision : 0.4
Résultat du context_recall : 0.3

% de réponses correctes : Eval Humaine
Incorrecte    0.5
Correcte      0.5
Name: proportion, dtype: float64

===== Résultats de l'évaluation ragas avec un seuil à 0.65, k = 3 et la version faiss_index_short =====
Résultat du faithfullness : 0.8722
Résultat du answer_relevancy : 0.878
Résultat du context_precision : 0.6833
Résultat du con